# Transformer NQS sweep — 2D toric code (L=4, OBC)

Minimal `nqs_sweep`-style notebook: every training cell is a **command line**
(`!python -m model.train_2d ...`) that **streams every step live** (energy,
`delta` vs exact diagonalization, V-score) and logs to **Weights & Biases**.

At L=4 OBC (N=24 = 2^24 states) we get the exact reference `E_exact` once with a
**Krylov/Lanczos** diagonalization, then pass it to each run via `--exact_E0`.

**Before running:** push the branch with `model/{transformer,builders_2d,train_2d}.py`,
set `REPO_URL`/`BRANCH` below, and pick a **GPU** runtime (Runtime → Change runtime type).

## 1. Setup — install, clone, W&B login

In [ ]:
!pip -q install netket wandb optax numba scipy

import os, sys
# --- set me -----------------------------------------------------------------
REPO_URL = "https://github.com/<YOUR_USER>/Approximate-Symmetries-TC.git"  # PAT-in-URL if private
BRANCH   = "nersc-gridinv-runs"
# ---------------------------------------------------------------------------
REPO_DIR = "/content/Approximate-Symmetries-TC"
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git fetch origin $BRANCH && git checkout $BRANCH && git pull
%cd $REPO_DIR

import wandb; wandb.login()      # paste your API key (or use --no_wandb / --wandb_offline below)

## 2. Exact diagonalization — run once, sets `E_exact`

Matrix-free (Numba) Hamiltonian → `scipy.sparse.linalg.eigsh`. ~2–3 GB and a few
minutes at N=24. `E_exact` is substituted into the command cells below as
`--exact_E0={E_exact}`.

In [ ]:
import jax; jax.config.update("jax_enable_x64", True)
from model.geometry import ToricCodeGeometry
from model.exact_diag import hamiltonian_linop
from scipy.sparse.linalg import eigsh

L, BC, HX, HZ, J = 4, "OBC", 0.2, 0.2, 1.0
geo = ToricCodeGeometry(L, L, BC)
print("N =", geo.N, "qubits  (2^N =", 2**geo.N, "states)")

Hlin, _ = hamiltonian_linop(geo, hx=HX, hy=0.0, hz=HZ, J=J)
E_exact = float(eigsh(Hlin, k=1, which="SA", return_eigenvectors=False)[0])
print("E_exact =", E_exact)

## 3. Combo baseline (Approximately-Symmetric NQS)

A_v baked in via the Wilson product; ~hundreds of params (`qgt=auto` → dense SR).

In [ ]:
!python -m model.train_2d --arch Combo --L 4 --hx 0.2 --hz 0.2 --bc OBC \
  --exact_E0={E_exact} --n_iter 400 --dt 2e-2 --lr_min 2e-3 --diag_shift 2e-4 \
  --channels_noninv 1 16 --channels_inv 16 8 1 --kernel_size 2 \
  --wandb_group L4_OBC_hx0.2_hz0.2

## 4. Transformer — edit the flags to tune

Knobs: `--d` (residual-stream dim), `--n_heads`, `--n_layers`, `--mlp_ratio`,
`--dt` (lr), `--lr_min`, `--diag_shift`, `--n_iter`, `--n_samples`. `--qgt minsr`
(sample-space S-matrix trick) is right because n_params > n_samples.
**`--d` must be divisible by `--n_heads`.** Copy this cell and edit for each variant.

In [ ]:
!python -m model.train_2d --arch factored_transformer --L 4 --hx 0.2 --hz 0.2 --bc OBC \
  --exact_E0={E_exact} --n_iter 500 --dt 2e-2 --lr_min 2e-3 --diag_shift 1e-3 \
  --d 32 --n_heads 4 --n_layers 4 --mlp_ratio 2 --qgt minsr \
  --wandb_group L4_OBC_hx0.2_hz0.2

### Variant: smaller (d=16)

In [ ]:
!python -m model.train_2d --arch factored_transformer --L 4 --hx 0.2 --hz 0.2 --bc OBC \
  --exact_E0={E_exact} --n_iter 500 --dt 2e-2 --lr_min 2e-3 --diag_shift 1e-3 \
  --d 16 --n_heads 4 --n_layers 4 --qgt minsr \
  --wandb_group L4_OBC_hx0.2_hz0.2

### Variant: deeper/wider (d=48, 6 heads, 6 layers, lower lr)

In [ ]:
!python -m model.train_2d --arch factored_transformer --L 4 --hx 0.2 --hz 0.2 --bc OBC \
  --exact_E0={E_exact} --n_iter 600 --dt 1e-2 --lr_min 1e-3 --diag_shift 3e-3 \
  --d 48 --n_heads 6 --n_layers 6 --qgt minsr \
  --wandb_group L4_OBC_hx0.2_hz0.2

## Notes

- Every run streams `step  k/N:  E = ...  delta = ...` live and logs to W&B
  (project `approx-sym-2D-TC-transformer`, group `L4_OBC_hx0.2_hz0.2`). Add
  `--wandb_entity <you>` if the default entity isn't yours, or `--no_wandb`
  (prints only) / `--wandb_offline` (no network) for quick tests.
- Resume a timed-out run: re-run the same cell with `--resume`.
- On a GPU runtime `train_2d` auto-scales chains (1024) and doubles samples.
- OOM at large `--d`? add `--chunk_size 2048` or lower `--n_samples`.
- `delta` is the error vs ED; also compare the printed `n_params` to weigh
  accuracy against parameter cost (the Step-1 point: the transformer matches
  Combo's accuracy only at many more params).